In [ ]:
from anthropic import Anthropic
import json

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Generate mock user-prompts

def generate_mock_prompts():
    messages = [
        {
            "role": "user",
            "content": """
            Generate 3 mock user-prompts to test an AI powered diet assistant chatbot that provides nutritional information on any given food items.
            Each prompt should mimic a user informing the chatbot about one or more food items that they either consumed or are planning to consume.
            "This chatbot is primarily designed for Indian users. So the prompts should be in the context of Indian cuisine and food items."

            Here is an example response:
            <example_response>
            [
                "I just had 3 dosas with coconut chutney and a glass of mango lassi for breakfast",
                "I'm planning to have chicken biryani for lunch",
                "I had a chapati with chicken curry for dinner."
            ]
            </example_response>
            """
        },
        {
            "role": "assistant",
            "content": "Here are the mock user prompts in JSON format:\n```json"
        }
    ]
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        temperature=1.0,
        stop_sequences=["```"])

    return json.loads(response.content[0].text)


In [51]:
# Grade the AI response to the user prompt in a scale of 1 to 10, where 1 is the worst and 10 is the best.

def grade_response_with_ai(user_prompt, ai_response):
    prompt = """
        Grade the following AI response to the user prompt in a scale of 1 to 10, where 1 is the worst and 10 is the best:

        <user_prompt>
        {user_prompt}
        </user_prompt>

        <ai_response>
        {ai_response}
        </ai_response>

        The grading should be based on the following criteria:
        <criteria>
        1. Accuracy: How accurate is the information provided in the AI response?
        2. Relevance: How relevant is the AI response to the user prompt?
        3. The AI response should be in the context of Indian cuisine and food items.
        4. The AI response should provide nutritional information on the food items mentioned in the user prompt.
        5. The AI response should be in the following JSON format:
        [
            {{
                "food_item_1": {{
                    "calories": "value",
                    "protein": "value",
                    "fiber": "value",
                    "fats": "value",
                    "carbohydrates": "value"
                }},
                "food_item_2": {{
                    "calories": "value",
                    "protein": "value",
                    "fiber": "value",
                    "fats": "value",
                    "carbohydrates": "value"
                }}
            }}
        ]
        </criteria>

        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "grade": An integer from 1 to 10
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment

        Example response:
        <example_response>
        {{
            "grade": 8,
            "strengths": ["strength 1", "strength 2", "strength 3"],
            "weaknesses": ["weakness 1", "weakness 2", "weakness 3"],
            "reasoning": "provide a concise explanation of your overall assessment"
        }}
        </example_response>
    """.format(user_prompt=user_prompt, ai_response=ai_response)

    messages = [
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": "Here is the evaluation in JSON format:\n```json"
        }
    ]
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        temperature=1.0,
        stop_sequences=["```"])

    evaluation = json.loads(response.content[0].text)
    return ai_response, evaluation["grade"], evaluation["reasoning"]

In [ ]:
def grade_response_with_code(ai_response):
    try:
        json.loads(ai_response)
        return 1
    except (ValueError, TypeError):
        return 0


In [ ]:
mock_user_prompts = generate_mock_prompts()

In [ ]:
def get_messages_wo_prompt_engineering(user_prompt):
    return [
        {
            "role": "user",
            "content": f"What is in the following food items? {user_prompt}"
        }
    ]

In [46]:
def get_messages_with_prompt_engineering(user_prompt):
    return [
        {
            "role": "user",
            "content": """
                Respond as an expert nutritionist and dietician in the context of Indian cuisine and food items.

                Parse food items from the following text and breakdown it's

                total calories and it's nutritional content of protein, fiber, fats and carbohydrates.

                If you are approximating some nutritional values because the user had provided a vague such as, "one glass of milk", in this case you don't know how big of a glass or what type of milk, etc. Add the word approx as suffice to the nutitional value.

                <user_prompt>
                {user_prompt}
                </user_prompt>

                Return the response in JSON format.

                Here is an example of the expected response.

                <example_response>
                [
                    {{
                        "food_item_1": {{
                            "calories": "value",
                            "protein": "value",
                            "fiber": "value",
                            "fats": "value (approx)",
                            "carbohydrates": "value"
                        }},
                        "food_item_2": {{
                            "calories": "value",
                            "protein": "value",
                            "fiber": "value",
                            "fats": "value",
                            "carbohydrates": "value"
                        }},
                        ...,
                        "Total": {{
                            "calories": "value",
                            "protein": "value",
                            "fiber": "value",
                            "fats": "value",
                            "carbohydrates": "value"
                        }}
                    }}
                ]
                </<example_response>
            """.format(user_prompt=user_prompt)
        },
        {
            "role": "assistant",
            "content": "Here is the JSON response: \n```json"
        }
    ]

In [52]:
ai_evals = {}
code_evals = {}

for user_prompt in mock_user_prompts:

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=get_messages_wo_prompt_engineering(user_prompt),
        temperature=1.0,
        stop_sequences=["```"])

    ai_evals[user_prompt] = grade_response_with_ai(user_prompt, response.content[0].text)
    code_evals[user_prompt] = grade_response_with_code(response.content[0].text)

In [50]:
ai_scores = []
for user_prompt, eval in ai_evals.items():
    ai_response, grade, reasoning = eval
    print("\n")
    print("++++++++++++")
    print("User Prompt:")
    print(user_prompt)
    print("++++++++++++")
    print("AI response:")
    print(eval[0])
    print("++++++++++++")
    print("Grade:")
    print(eval[1])
    print("++++++++++++")
    print("Reasoning:")
    print(eval[2])
    print("++++++++++++")
    print("\n")
    ai_scores.append(eval[1])

final_scores = []
for ai_score, code_score in zip(ai_scores, code_evals.values()):
    print(f"ai_score: {ai_score}")
    print(f"code_score: {code_score}")
    final_scores.append(ai_score * code_score)

total_final_score = 0
for final_score in final_scores:
    total_final_score += final_score

score = total_final_score/len(final_scores)
print("\n")
print("++++++++")
print("Final Score:", score)
print("++++++++")



++++++++++++
User Prompt:
I just had 2 idlis with sambar and a cup of filter coffee for breakfast. How many calories did I consume?
++++++++++++
AI response:
# Nutritional Breakdown of Your Breakfast

## Individual Components:

**2 Idlis**
- Calories: ~140-160 kcal
- Carbs: 28-32g, Protein: 3-4g, Fat: 1-2g

**Sambar (1 cup)**
- Calories: ~80-120 kcal
- Carbs: 8-12g, Protein: 3-4g, Fat: 2-3g
- Contains: lentils, vegetables (typically okra, carrot, onion), spices, tamarind

**Filter Coffee (1 cup)**
- Calories: ~20-40 kcal (if black)
- If made with milk & sugar: ~60-100 kcal

## **Total Estimate: 240-420 kcal**

The variation depends on:
- **Idli size** (small vs. large)
- **Sambar preparation** (oil/ghee amount)
- **Coffee preparation** (milk ratio, sugar)

## What You Consumed:
- **Carbs-heavy** breakfast (balanced energy)
- **Moderate protein** (could add more with extra sambar or egg)
- **Low fat** (healthy option)
- **Rich in fiber & minerals** from lentils and vegetables

This is